# LSTM v3 híbrida residual — Recomendación de horarios

La evaluación de la v2 demostró que el promedio por día/hora (`MAE 18.452`) superó a la LSTM pura (`MAE 21.349`). Esta versión conserva ese resultado en lugar de ocultarlo y plantea una arquitectura híbrida:

`predicción final = patrón horario suavizado + corrección secuencial LSTM`

La LSTM analiza las últimas publicaciones y aprende si el rendimiento reciente justifica subir o bajar la expectativa normal de una franja. Si la secuencia no aporta, la corrección se reduce mediante un peso elegido exclusivamente con validación.

## Reglas metodológicas

- Se conserva la v2; esta es una nueva línea experimental.
- 70 % entrenamiento, 15 % validación y 15 % prueba en orden cronológico.
- El suavizado, los escaladores, la ventana y el peso híbrido se deciden sin mirar prueba.
- Para entrenamiento se usa un promedio *leave-one-out*: la publicación objetivo nunca participa en su propia referencia horaria.
- Se comparan ventanas de 3, 5, 7 y 14 publicaciones.
- Prueba se evalúa una única vez después de seleccionar la configuración.
- El piloto de integración solo se recomienda si el híbrido con aporte LSTM supera al mejor baseline en prueba interna.

> El CSV actual permite un modelo general de Facebook. No permite afirmar personalización por cuenta ni entrenamiento para Instagram. Además, la franja final de este CSV ya fue observada durante la evaluación de la v2: la prueba de v3 es una comprobación interna, no una confirmación externa completamente inédita. La validación definitiva deberá ser prospectiva con publicaciones nuevas.

In [ ]:
!pip install -q joblib seaborn
print("Dependencias listas.")

In [ ]:
import json
import os
import random
import shutil
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
import tensorflow as tf
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from tensorflow.keras import Model, layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

SEED = 623
MODEL_VERSION = "facebook_lstm_hybrid_v3.0.0"
EPOCHS = 120
BATCH_SIZE = 32
ARTIFACT_DIR = Path("/content/lstm_horarios_v3_artifacts")
WORK_DIR = Path("/content/lstm_horarios_v3_work")

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass
warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)
print("TensorFlow:", tf.__version__)
print("GPU disponible:", bool(tf.config.list_physical_devices("GPU")))

## 1. Subir y validar `dataset_facebook.csv`

In [ ]:
from google.colab import files

EXPECTED_FILENAME = "dataset_facebook.csv"
dataset_path = Path("/content") / EXPECTED_FILENAME
if not dataset_path.exists():
    print(f"Selecciona {EXPECTED_FILENAME}")
    uploaded = files.upload()
    if EXPECTED_FILENAME not in uploaded:
        if len(uploaded) != 1:
            raise ValueError(f"Sube un único archivo llamado {EXPECTED_FILENAME}.")
        Path("/content", next(iter(uploaded))).replace(dataset_path)
print("Dataset:", dataset_path, f"({dataset_path.stat().st_size:,} bytes)")

In [ ]:
REQUIRED_COLUMNS = [
    "fecha_publicacion", "hora_publicacion", "dia_semana",
    "reacciones", "comentarios", "clicks",
    "usuarios_interactuaron", "engagement_score",
]
NUMERIC_COLUMNS = REQUIRED_COLUMNS[1:]
df = pd.read_csv(dataset_path)
missing = sorted(set(REQUIRED_COLUMNS) - set(df.columns))
if missing:
    raise ValueError(f"Faltan columnas obligatorias: {missing}")
df = df[REQUIRED_COLUMNS].copy()
df["fecha_publicacion"] = pd.to_datetime(df["fecha_publicacion"], errors="coerce")
for column in NUMERIC_COLUMNS:
    df[column] = pd.to_numeric(df[column], errors="coerce")
invalid = int(df[REQUIRED_COLUMNS].isna().any(axis=1).sum())
if invalid:
    print(f"Se eliminarán {invalid} filas inválidas.")
df = df.dropna(subset=REQUIRED_COLUMNS).drop_duplicates().copy()
if not df["hora_publicacion"].between(0, 23).all():
    raise ValueError("hora_publicacion debe estar entre 0 y 23.")
if (df[["reacciones", "comentarios", "clicks", "usuarios_interactuaron", "engagement_score"]] < 0).any().any():
    raise ValueError("Las métricas no pueden ser negativas.")
df["timestamp"] = df["fecha_publicacion"].dt.normalize() + pd.to_timedelta(df["hora_publicacion"], unit="h")
calendar_day = df["timestamp"].dt.dayofweek
day_mismatches = int((df["dia_semana"].astype(int) != calendar_day).sum())
if day_mismatches:
    print(f"Se corrigieron {day_mismatches} días inconsistentes usando la fecha.")
df["dia_semana"] = calendar_day.astype(int)
calculated_score = df["reacciones"] + 2 * df["comentarios"] + df["clicks"]
score_mismatches = int((df["engagement_score"] != calculated_score).sum())
if score_mismatches:
    print(f"Se recalcularon {score_mismatches} scores inconsistentes.")
df["engagement_score"] = calculated_score.astype(float)
df = df.sort_values("timestamp", kind="stable").reset_index(drop=True)
if len(df) < 150:
    raise ValueError("Se requieren al menos 150 registros para esta comparación.")
print(f"Registros: {len(df):,}")
print("Periodo:", df["timestamp"].min(), "→", df["timestamp"].max())
display(df.head())

## 2. Reservar los periodos antes de cualquier ajuste

Los límites se fijan por filas cronológicas. La prueba permanece aislada hasta la evaluación final.

In [ ]:
N = len(df)
TRAIN_END = int(N * 0.70)
VAL_END = int(N * 0.85)
df["target_log"] = np.log1p(df["engagement_score"].astype(float))

split_summary = pd.DataFrame([
    {"conjunto": "Entrenamiento", "filas": TRAIN_END, "desde": df.iloc[0]["timestamp"], "hasta": df.iloc[TRAIN_END - 1]["timestamp"]},
    {"conjunto": "Validación", "filas": VAL_END - TRAIN_END, "desde": df.iloc[TRAIN_END]["timestamp"], "hasta": df.iloc[VAL_END - 1]["timestamp"]},
    {"conjunto": "Prueba", "filas": N - VAL_END, "desde": df.iloc[VAL_END]["timestamp"], "hasta": df.iloc[-1]["timestamp"]},
])
display(split_summary)

## 3. Elegir el suavizado horario usando solo validación

Una franja con pocas muestras se acerca a la media global. Se prueban diferentes intensidades de suavizado y se selecciona la de menor MAE en validación.

In [ ]:
train_df = df.iloc[:TRAIN_END].copy()
val_df = df.iloc[TRAIN_END:VAL_END].copy()
test_df = df.iloc[VAL_END:].copy()
global_train_log = float(train_df["target_log"].mean())
global_train_score = float(train_df["engagement_score"].mean())
slot_stats = train_df.groupby(["dia_semana", "hora_publicacion"])["target_log"].agg(["sum", "count"])
raw_slot_means = train_df.groupby(["dia_semana", "hora_publicacion"])["engagement_score"].mean().to_dict()

def full_slot_prior_log(day, hour, strength):
    key = (int(day), int(hour))
    if key not in slot_stats.index:
        return global_train_log
    row = slot_stats.loc[key]
    return float((row["sum"] + strength * global_train_log) / (row["count"] + strength))

def predict_baseline(rows, strength):
    logs = np.array([full_slot_prior_log(row.dia_semana, row.hora_publicacion, strength) for row in rows.itertuples()])
    return np.maximum(np.expm1(logs), 0.0)

smoothing_rows = []
for strength in [0.5, 1, 2, 3, 5, 8, 12, 20]:
    prediction = predict_baseline(val_df, strength)
    smoothing_rows.append({
        "smoothing_strength": float(strength),
        "validation_MAE": float(mean_absolute_error(val_df["engagement_score"], prediction)),
        "validation_RMSE": float(np.sqrt(mean_squared_error(val_df["engagement_score"], prediction))),
    })
smoothing_results = pd.DataFrame(smoothing_rows).sort_values("validation_MAE").reset_index(drop=True)
SMOOTHING_STRENGTH = float(smoothing_results.iloc[0]["smoothing_strength"])
display(smoothing_results.style.format({"validation_MAE": "{:.3f}", "validation_RMSE": "{:.3f}"}))
print("Suavizado seleccionado sin usar prueba:", SMOOTHING_STRENGTH)

## 4. Construir residuos sin autorreferencia

Para una fila de entrenamiento, su referencia se calcula retirando esa misma fila del grupo y de la media global. Validación y prueba usan exclusivamente estadísticas de entrenamiento.

In [ ]:
train_global_sum = float(train_df["target_log"].sum())
baseline_log = np.empty(N, dtype=np.float32)
slot_count = np.empty(N, dtype=np.float32)

for index, row in df.iterrows():
    key = (int(row["dia_semana"]), int(row["hora_publicacion"]))
    if index < TRAIN_END:
        group = slot_stats.loc[key]
        count_without_row = max(int(group["count"]) - 1, 0)
        global_without_row = (train_global_sum - float(row["target_log"])) / max(TRAIN_END - 1, 1)
        baseline_log[index] = (
            float(group["sum"]) - float(row["target_log"]) + SMOOTHING_STRENGTH * global_without_row
        ) / (count_without_row + SMOOTHING_STRENGTH)
        slot_count[index] = count_without_row
    else:
        baseline_log[index] = full_slot_prior_log(row["dia_semana"], row["hora_publicacion"], SMOOTHING_STRENGTH)
        slot_count[index] = float(slot_stats.loc[key, "count"]) if key in slot_stats.index else 0.0

df["baseline_log"] = baseline_log
df["residual_log"] = df["target_log"] - df["baseline_log"]
df["slot_count_log"] = np.log1p(slot_count)
print("Media del residuo en entrenamiento:", round(float(df.iloc[:TRAIN_END]["residual_log"].mean()), 6))
display(df[["timestamp", "engagement_score", "baseline_log", "residual_log", "slot_count_log"]].head())

## 5. Características históricas y del candidato

In [ ]:
df["hora_sin"] = np.sin(2 * np.pi * df["hora_publicacion"] / 24)
df["hora_cos"] = np.cos(2 * np.pi * df["hora_publicacion"] / 24)
df["dia_sin"] = np.sin(2 * np.pi * df["dia_semana"] / 7)
df["dia_cos"] = np.cos(2 * np.pi * df["dia_semana"] / 7)
df["gap_hours"] = df["timestamp"].diff().dt.total_seconds().div(3600).clip(lower=0, upper=24 * 30).fillna(24.0)
for source, destination in {
    "reacciones": "reacciones_log", "comentarios": "comentarios_log",
    "clicks": "clicks_log", "usuarios_interactuaron": "usuarios_log",
    "engagement_score": "engagement_log", "gap_hours": "gap_log",
}.items():
    df[destination] = np.log1p(df[source].astype(float))

HISTORY_FEATURES = [
    "reacciones_log", "comentarios_log", "clicks_log", "usuarios_log",
    "engagement_log", "residual_log", "hora_sin", "hora_cos",
    "dia_sin", "dia_cos", "gap_log",
]
CANDIDATE_FEATURES = [
    "hora_sin", "hora_cos", "dia_sin", "dia_cos", "gap_log",
    "baseline_log", "slot_count_log",
]
history_raw = df[HISTORY_FEATURES].to_numpy(dtype=np.float32)
candidate_raw = df[CANDIDATE_FEATURES].to_numpy(dtype=np.float32)
residual_raw = df[["residual_log"]].to_numpy(dtype=np.float32)
print("Histórico:", HISTORY_FEATURES)
print("Candidato:", CANDIDATE_FEATURES)

## 6. Entrenar candidatos usando solamente entrenamiento y validación

Se prueban cuatro ventanas. Para cada modelo, el peso `alpha` de la corrección LSTM se elige por MAE de validación entre 0 y 1. `alpha=0` significa que la secuencia no añadió valor; `alpha=1` aplica toda la corrección.

In [ ]:
CONFIGS = [
    {"window": 3, "units": 24, "dropout": 0.10},
    {"window": 5, "units": 32, "dropout": 0.15},
    {"window": 7, "units": 32, "dropout": 0.20},
    {"window": 14, "units": 40, "dropout": 0.20},
]
ALPHA_GRID = np.round(np.linspace(0.0, 1.0, 21), 2)

def build_sequences(window):
    histories, candidates, targets, indices = [], [], [], []
    for target_index in range(window, N):
        histories.append(history_raw[target_index - window:target_index])
        candidates.append(candidate_raw[target_index])
        targets.append(residual_raw[target_index])
        indices.append(target_index)
    return (np.asarray(histories, np.float32), np.asarray(candidates, np.float32),
            np.asarray(targets, np.float32), np.asarray(indices, np.int32))

def make_model(window, units, dropout):
    tf.keras.backend.clear_session()
    history_input = layers.Input((window, len(HISTORY_FEATURES)), name="history_sequence")
    h = layers.LSTM(units, dropout=dropout, recurrent_dropout=0.0, name="history_lstm")(history_input)
    h = layers.Dense(20, activation="relu")(h)
    candidate_input = layers.Input((len(CANDIDATE_FEATURES),), name="candidate_slot")
    c = layers.Dense(12, activation="relu")(candidate_input)
    x = layers.Concatenate()([h, c])
    x = layers.Dense(24, activation="relu")(x)
    x = layers.Dropout(dropout)(x)
    output = layers.Dense(1, name="predicted_residual")(x)
    result = Model([history_input, candidate_input], output, name=f"residual_lstm_w{window}")
    result.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="mse", metrics=["mae"])
    return result

def inverse_residual(scaler, values):
    return scaler.inverse_transform(np.asarray(values).reshape(-1, 1)).ravel()

experiments = []
for config_number, config in enumerate(CONFIGS, start=1):
    window = config["window"]
    print(f"\n[{config_number}/{len(CONFIGS)}] Entrenando ventana={window}...")
    Xh_raw, Xc_raw, yr_raw, indices = build_sequences(window)
    train_mask = indices < TRAIN_END
    val_mask = (indices >= TRAIN_END) & (indices < VAL_END)
    test_mask = indices >= VAL_END
    scaler_history = StandardScaler().fit(history_raw[:TRAIN_END])
    scaler_candidate = StandardScaler().fit(candidate_raw[:TRAIN_END])
    scaler_residual = StandardScaler().fit(yr_raw[train_mask])
    shape = Xh_raw.shape
    Xh = scaler_history.transform(Xh_raw.reshape(-1, shape[-1])).reshape(shape).astype(np.float32)
    Xc = scaler_candidate.transform(Xc_raw).astype(np.float32)
    yr = scaler_residual.transform(yr_raw).astype(np.float32)
    model = make_model(**config)
    callbacks = [
        EarlyStopping(monitor="val_loss", patience=14, min_delta=1e-4, restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=6, min_lr=1e-5, verbose=0),
    ]
    history = model.fit(
        {"history_sequence": Xh[train_mask], "candidate_slot": Xc[train_mask]}, yr[train_mask],
        validation_data=({"history_sequence": Xh[val_mask], "candidate_slot": Xc[val_mask]}, yr[val_mask]),
        epochs=EPOCHS, batch_size=BATCH_SIZE, shuffle=False, callbacks=callbacks, verbose=0,
    )
    val_residual_scaled = model.predict(
        {"history_sequence": Xh[val_mask], "candidate_slot": Xc[val_mask]}, verbose=0
    )
    val_residual_prediction = inverse_residual(scaler_residual, val_residual_scaled)
    val_indices = indices[val_mask]
    val_actual = df.iloc[val_indices]["engagement_score"].to_numpy(float)
    val_base_log = df.iloc[val_indices]["baseline_log"].to_numpy(float)
    alpha_scores = []
    for alpha in ALPHA_GRID:
        prediction = np.maximum(np.expm1(val_base_log + alpha * val_residual_prediction), 0.0)
        alpha_scores.append((float(alpha), float(mean_absolute_error(val_actual, prediction))))
    best_alpha, best_val_mae = min(alpha_scores, key=lambda item: item[1])
    experiments.append({
        "config": config, "model": model, "history": history.history,
        "scaler_history": scaler_history, "scaler_candidate": scaler_candidate,
        "scaler_residual": scaler_residual, "Xh": Xh, "Xc": Xc, "yr": yr,
        "indices": indices, "train_mask": train_mask, "val_mask": val_mask, "test_mask": test_mask,
        "alpha": best_alpha, "validation_mae": best_val_mae,
        "best_epoch": int(np.argmin(history.history["val_loss"]) + 1),
    })
    print(f"Ventana {window}: alpha={best_alpha:.2f}, MAE validación={best_val_mae:.3f}")

In [ ]:
validation_results = pd.DataFrame([{
    "window": item["config"]["window"],
    "units": item["config"]["units"],
    "dropout": item["config"]["dropout"],
    "alpha": item["alpha"],
    "best_epoch": item["best_epoch"],
    "validation_MAE": item["validation_mae"],
} for item in experiments]).sort_values("validation_MAE").reset_index(drop=True)
best_experiment = min(experiments, key=lambda item: item["validation_mae"])
display(validation_results.style.format({"alpha": "{:.2f}", "validation_MAE": "{:.3f}"}))
print("Configuración seleccionada sin usar prueba:", best_experiment["config"])
print("Peso LSTM seleccionado:", best_experiment["alpha"])

## 7. Evaluación final: abrir el conjunto de prueba una sola vez

In [ ]:
selected = best_experiment
test_mask = selected["test_mask"]
test_indices = selected["indices"][test_mask]
actual_test = df.iloc[test_indices]["engagement_score"].to_numpy(float)
test_base_log = df.iloc[test_indices]["baseline_log"].to_numpy(float)
test_residual_scaled = selected["model"].predict(
    {"history_sequence": selected["Xh"][test_mask], "candidate_slot": selected["Xc"][test_mask]}, verbose=0
)
test_residual_prediction = inverse_residual(selected["scaler_residual"], test_residual_scaled)
pred_hybrid = np.maximum(np.expm1(test_base_log + selected["alpha"] * test_residual_prediction), 0.0)
pred_smoothed = np.maximum(np.expm1(test_base_log), 0.0)
pred_raw_slot = np.array([
    raw_slot_means.get((int(row.dia_semana), int(row.hora_publicacion)), global_train_score)
    for row in df.iloc[test_indices].itertuples()
])
pred_global = np.full_like(actual_test, global_train_score)

def regression_metrics(actual, predicted):
    correlation = spearmanr(actual, predicted).statistic
    return {
        "MAE": float(mean_absolute_error(actual, predicted)),
        "RMSE": float(np.sqrt(mean_squared_error(actual, predicted))),
        "R2": float(r2_score(actual, predicted)),
        "Spearman": float(correlation) if np.isfinite(correlation) else 0.0,
    }

metric_rows = []
for name, prediction in {
    "Media global": pred_global,
    "Promedio día/hora v2": pred_raw_slot,
    "Promedio suavizado v3": pred_smoothed,
    "Híbrido residual LSTM v3": pred_hybrid,
}.items():
    metric_rows.append({"modelo": name, **regression_metrics(actual_test, prediction)})
metrics_df = pd.DataFrame(metric_rows).sort_values("MAE").reset_index(drop=True)
display(metrics_df.style.format({"MAE": "{:.3f}", "RMSE": "{:.3f}", "R2": "{:.3f}", "Spearman": "{:.3f}"}))

hybrid_mae = float(metrics_df.loc[metrics_df["modelo"] == "Híbrido residual LSTM v3", "MAE"].iloc[0])
best_baseline_mae = float(metrics_df.loc[metrics_df["modelo"] != "Híbrido residual LSTM v3", "MAE"].min())
improvement = (best_baseline_mae - hybrid_mae) / best_baseline_mae * 100
APPROVED_FOR_PILOT = bool(selected["alpha"] > 0 and hybrid_mae < best_baseline_mae)
if APPROVED_FOR_PILOT:
    print(f"APROBADO PARA PILOTO: el híbrido reduce el MAE {improvement:.2f}% y alpha={selected['alpha']:.2f} confirma aporte LSTM.")
    print("La confirmación final será prospectiva con publicaciones nuevas no presentes en este CSV.")
else:
    print(f"NO APROBADO todavía: mejora frente al mejor baseline={improvement:.2f}%, alpha={selected['alpha']:.2f}.")
    print("Conserva estas métricas; será necesario enriquecer datos o revisar características antes de afirmar superioridad.")

In [ ]:
comparison = pd.DataFrame({
    "timestamp": df.iloc[test_indices]["timestamp"].to_numpy(),
    "real": actual_test, "hibrido_lstm": pred_hybrid,
    "promedio_suavizado": pred_smoothed, "promedio_v2": pred_raw_slot,
})
training_history = pd.DataFrame(selected["history"])
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].plot(training_history["loss"], label="Entrenamiento")
axes[0].plot(training_history["val_loss"], label="Validación")
axes[0].axvline(selected["best_epoch"] - 1, color="#ef6c22", linestyle="--")
axes[0].set_title("Entrenamiento del modelo seleccionado")
axes[0].legend()
axes[1].plot(comparison["timestamp"], comparison["real"], label="Real", alpha=0.75)
axes[1].plot(comparison["timestamp"], comparison["hibrido_lstm"], label="Híbrido LSTM")
axes[1].plot(comparison["timestamp"], comparison["promedio_suavizado"], label="Baseline", alpha=0.7)
axes[1].set_title("Prueba cronológica")
axes[1].tick_params(axis="x", rotation=30)
axes[1].legend()
plt.tight_layout()
plt.show()
display(comparison.head(20))

## 8. Ranking futuro con el modelo seleccionado

El ranking muestra por separado el patrón horario, la corrección de la LSTM y la predicción final. Esto permitirá explicar la recomendación dentro de Laravel.

In [ ]:
DAY_NAMES = {0: "Lunes", 1: "Martes", 2: "Miércoles", 3: "Jueves", 4: "Viernes", 5: "Sábado", 6: "Domingo"}
observed_hours = sorted(train_df["hora_publicacion"].astype(int).unique().tolist())
window = selected["config"]["window"]
latest_history = df.iloc[-window:].copy()
last_timestamp = pd.Timestamp(df.iloc[-1]["timestamp"])

def make_candidate_row(timestamp, last_timestamp):
    day, hour = timestamp.dayofweek, timestamp.hour
    gap = min(max((timestamp - last_timestamp).total_seconds() / 3600, 0), 24 * 30)
    prior = full_slot_prior_log(day, hour, SMOOTHING_STRENGTH)
    key = (int(day), int(hour))
    count = float(slot_stats.loc[key, "count"]) if key in slot_stats.index else 0.0
    features = [
        np.sin(2 * np.pi * hour / 24), np.cos(2 * np.pi * hour / 24),
        np.sin(2 * np.pi * day / 7), np.cos(2 * np.pi * day / 7),
        np.log1p(gap), prior, np.log1p(count),
    ]
    return features, prior, count

candidate_records = []
for offset in range(8):
    day_date = last_timestamp.normalize() + pd.Timedelta(days=offset)
    for hour in observed_hours:
        timestamp = day_date + pd.Timedelta(hours=hour)
        if timestamp <= last_timestamp:
            continue
        features, prior, count = make_candidate_row(timestamp, last_timestamp)
        candidate_records.append({"timestamp": timestamp, "features": features, "prior_log": prior, "samples": count})

candidate_matrix = np.asarray([item["features"] for item in candidate_records], np.float32)
candidate_scaled = selected["scaler_candidate"].transform(candidate_matrix).astype(np.float32)
history_matrix = latest_history[HISTORY_FEATURES].to_numpy(np.float32)[None, :, :]
shape = history_matrix.shape
history_scaled = selected["scaler_history"].transform(history_matrix.reshape(-1, shape[-1])).reshape(shape).astype(np.float32)
history_repeated = np.repeat(history_scaled, len(candidate_records), axis=0)
residual_scaled = selected["model"].predict(
    {"history_sequence": history_repeated, "candidate_slot": candidate_scaled}, verbose=0
)
residual_predictions = inverse_residual(selected["scaler_residual"], residual_scaled)
ranking_rows = []
for item, residual in zip(candidate_records, residual_predictions):
    final_score = max(np.expm1(item["prior_log"] + selected["alpha"] * residual), 0.0)
    ranking_rows.append({
        "timestamp": item["timestamp"], "dia": DAY_NAMES[item["timestamp"].dayofweek],
        "dia_semana": int(item["timestamp"].dayofweek), "hora": f"{item['timestamp'].hour:02d}:00",
        "muestras_franja": int(item["samples"]),
        "score_base": float(np.expm1(item["prior_log"])),
        "correccion_lstm_log": float(selected["alpha"] * residual),
        "engagement_predicho": float(final_score),
    })
future_ranking = pd.DataFrame(ranking_rows).sort_values("engagement_predicho", ascending=False).reset_index(drop=True)
display(future_ranking.head(10).style.format({"score_base": "{:.2f}", "correccion_lstm_log": "{:.4f}", "engagement_predicho": "{:.2f}"}))

## 9. Exportar modelo, contrato y evidencias

In [ ]:
model_path = ARTIFACT_DIR / "modelo_lstm_residual_v3.keras"
selected["model"].save(model_path)
joblib.dump(selected["scaler_history"], ARTIFACT_DIR / "scaler_history.joblib")
joblib.dump(selected["scaler_candidate"], ARTIFACT_DIR / "scaler_candidate.joblib")
joblib.dump(selected["scaler_residual"], ARTIFACT_DIR / "scaler_residual.joblib")
validation_results.to_csv(ARTIFACT_DIR / "seleccion_validacion.csv", index=False)
smoothing_results.to_csv(ARTIFACT_DIR / "seleccion_suavizado.csv", index=False)
metrics_df.to_csv(ARTIFACT_DIR / "metricas_prueba.csv", index=False)
comparison.to_csv(ARTIFACT_DIR / "predicciones_prueba.csv", index=False)
future_ranking.to_csv(ARTIFACT_DIR / "ranking_demostracion.csv", index=False)
pd.DataFrame(selected["history"]).to_csv(ARTIFACT_DIR / "historial_entrenamiento.csv", index=False)

slot_prior_export = []
for (day, hour), row in slot_stats.iterrows():
    prior_log = full_slot_prior_log(day, hour, SMOOTHING_STRENGTH)
    slot_prior_export.append({
        "dia_semana": int(day), "hora": int(hour), "samples": int(row["count"]),
        "prior_log": float(prior_log), "prior_score": float(np.expm1(prior_log)),
    })
with open(ARTIFACT_DIR / "slot_priors.json", "w", encoding="utf-8") as file:
    json.dump(slot_prior_export, file, ensure_ascii=False, indent=2)

metadata = {
    "model_version": MODEL_VERSION, "platform": "facebook",
    "architecture": "smoothed slot prior + weighted residual LSTM",
    "window": int(window), "alpha": float(selected["alpha"]),
    "smoothing_strength": float(SMOOTHING_STRENGTH),
    "history_features": HISTORY_FEATURES, "candidate_features": CANDIDATE_FEATURES,
    "target": "log1p(engagement_score) - smoothed_slot_prior_log",
    "engagement_formula": "reacciones + 2 * comentarios + clicks",
    "timezone_expected": "America/La_Paz", "day_convention": "0=lunes, 6=domingo",
    "global_train_log": global_train_log, "global_train_score": global_train_score,
    "observed_hours": observed_hours, "approved_for_pilot": APPROVED_FOR_PILOT,
    "test_improvement_percent": float(improvement),
    "dataset": {"rows": int(N), "start": df.iloc[0]["timestamp"].isoformat(), "end": df.iloc[-1]["timestamp"].isoformat()},
    "split": {"train_end": TRAIN_END, "validation_end": VAL_END, "strategy": "chronological"},
    "limitations": [
        "Sin account_id: modelo general, no personalizado durante entrenamiento.",
        "Dataset de Facebook; no afirmar entrenamiento para Instagram.",
        "Sin reach: objetivo absoluto, no engagement rate.",
    ],
}
with open(ARTIFACT_DIR / "metadata.json", "w", encoding="utf-8") as file:
    json.dump(metadata, file, ensure_ascii=False, indent=2)

print("Artefactos:")
for path in sorted(ARTIFACT_DIR.iterdir()):
    print(" -", path.name, f"({path.stat().st_size:,} bytes)")

In [ ]:
archive_path = shutil.make_archive("/content/lstm_horarios_facebook_v3_hibrido", "zip", root_dir=ARTIFACT_DIR)
print("Paquete final:", archive_path)
print("Tamaño:", f"{Path(archive_path).stat().st_size / 1024 / 1024:.2f} MB")
files.download(archive_path)

## Resultado que debes compartir antes de integrar

Copia o captura:

1. Tabla `validation_results`.
2. Tabla final `metrics_df`.
3. Mensaje `APROBADO PARA PILOTO` o `NO APROBADO`.
4. Primeras diez filas de `future_ranking`.
5. Conserva `lstm_horarios_facebook_v3_hibrido.zip`.

No se debe modificar el umbral después de conocer esta prueba interna. Si el resultado es favorable, se puede desplegar un piloto y medir publicaciones nuevas; si es desfavorable, el siguiente paso será enriquecer el dataset, no ajustar repetidamente contra la misma prueba.